In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Online Orders - In-Memory Pipeline") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()


In [2]:
orders_data = [
("O001","Delhi ","Laptop","45000","2024-01-05","Completed"),
("O002","Mumbai","Mobile ","32000","05/01/2024","Completed"),
("O003","Bangalore","Tablet","30000","2024/01/06","Completed"),
("O004","Delhi","Laptop","","2024-01-07","Cancelled"),
("O005","Mumbai","Mobile","invalid","2024-01-08","Completed"),
("O006","Chennai","Tablet",None,"2024-01-08","Completed"),
("O007","Delhi","Laptop","47000","09-01-2024","Completed"),
("O008","Bangalore","Mobile","28000","2024-01-09","Completed"),
("O009","Mumbai","Laptop","55000","2024-01-10","Completed"),
("O009","Mumbai","Laptop","55000","2024-01-10","Completed")
]

PHASE 1 — DATA INGESTION & SCHEMA

1. Define explicit schema

In [3]:

orders_schema = StructType([
    StructField("order_id",   StringType(), True),
    StructField("city",       StringType(), True),
    StructField("product",    StringType(), True),
    StructField("amount",     StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status",     StringType(), True),
])


2) Create a DataFrame using the schema


In [4]:
orders_raw = spark.createDataFrame(orders_data, schema=orders_schema)

3) Print schema and validate

In [5]:
orders_raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)



PHASE 2 — DATA CLEANING

Trim all string columns; standardize city/product

In [6]:

clean1 = (orders_raw
    .withColumn("order_id",   F.trim("order_id"))
    .withColumn("city",       F.upper(F.trim("city")))      # UPPER for joins/partitioning
    .withColumn("product",    F.initcap(F.trim("product"))) # Proper case (e.g., 'Laptop', 'Mobile')
    .withColumn("status_raw", F.upper(F.trim("status")))    # normalize for filtering
    .withColumn("amount_str", F.trim("amount"))
    .withColumn("order_date_str", F.trim("order_date"))
)


Standardize status values (prefix-based)

In [8]:

clean2 = clean1.withColumn(
    "status",
    F.when(F.col("status_raw").startswith("COMP"), F.lit("COMPLETED"))
     .when(F.col("status_raw").startswith("CANC"), F.lit("CANCELLED"))
     .otherwise(F.lit("UNKNOWN"))
).drop("status_raw")


  Convert amount to IntegerType with validation

In [10]:

clean3 = clean2.withColumn(
    "amount_int",
    F.when(F.col("amount_str").rlike("^[0-9]+$"), F.col("amount_str").cast(IntegerType()))
     .otherwise(F.lit(None).cast(IntegerType()))
).drop("amount_str")
invalid_amount_df = clean3.filter(F.col("amount_int").isNull())


Parse multiple date formats into DateType

In [29]:

parsed_date = F.coalesce(
    F.try_to_timestamp(F.col("order_date_str"), F.lit('yyyy-MM-dd')).cast(DateType()),
    F.try_to_timestamp(F.col("order_date_str"), F.lit('dd/MM/yyyy')).cast(DateType()),
    F.try_to_timestamp(F.col("order_date_str"), F.lit('yyyy/MM/dd')).cast(DateType()),
    F.try_to_timestamp(F.col("order_date_str"), F.lit('dd-MM-yyyy')).cast(DateType())
)

clean4 = (clean3
    .withColumn("order_date_parsed", parsed_date)
    .drop("order_date_str", "order_date")
)
invalid_date_df = clean4.filter(F.col("order_date_parsed").isNull())


Remove duplicate orders

In [15]:
clean5 = clean4.dropDuplicates(["order_id"])

Keep only Completed orders and valid amounts/dates

In [16]:

cleaned_orders = (clean5
    .filter(F.col("status") == "COMPLETED")
    .filter(F.col("amount_int").isNotNull())
    .filter(F.col("order_date_parsed").isNotNull())
    .select(
        "order_id",
        "city",
        "product",
        F.col("amount_int").alias("amount"),
        F.col("order_date_parsed").alias("order_date"),
        "status",
    )
)


Phase 3
Total revenue per city

In [17]:

rev_by_city = cleaned_orders.groupBy("city").agg(
    F.sum("amount").alias("total_revenue")
)


Total revenue per product

In [18]:

rev_by_product = cleaned_orders.groupBy("product").agg(
    F.sum("amount").alias("total_revenue")
)


Average order value per city

In [19]:

aov_by_city = cleaned_orders.groupBy("city").agg(
    F.avg("amount").alias("avg_order_value")
)


Phase 4
Rank cities by total revenue

In [20]:

city_rank = rev_by_city.select(
    "city", "total_revenue",
    F.dense_rank().over(Window.orderBy(F.desc("total_revenue"))).alias("city_rank")
)


Identify top-performing city

In [21]:
top_city = city_rank.filter(F.col("city_rank") == 1)

Phase 5
Cache the cleaned DataFrame

In [ ]:
spark.catalog.clearCache()
cleaned_orders.cache()
cleaned_orders.count()


Run two aggregations and observe behavior (plans/runtime)

In [24]:

rev_by_city_cached = cleaned_orders.groupBy("city").agg(F.sum("amount").alias("total_revenue"))
aov_by_city_cached = cleaned_orders.groupBy("city").agg(F.avg("amount").alias("avg_order_value"))


Inspect plans (look for reduced scans/exchanges due to caching)

In [25]:

cleaned_orders.explain(True)
rev_by_city_cached.explain(True)
aov_by_city_cached.explain(True)


== Parsed Logical Plan ==
'Project ['order_id, 'city, 'product, 'amount_int AS amount#17, 'order_date_parsed AS order_date#18, 'status]
+- Filter isnotnull(order_date_parsed#16)
   +- Filter isnotnull(amount_int#15)
      +- Filter (status#13 = COMPLETED)
         +- Deduplicate [order_id#6]
            +- Project [order_id#6, city#7, product#8, amount#3, status#13, amount_int#15, order_date_parsed#16]
               +- Project [order_id#6, city#7, product#8, amount#3, order_date#4, status#13, order_date_str#11, amount_int#15, coalesce(to_date(order_date_str#11, Some(yyyy-MM-dd), Some(Etc/UTC), true), to_date(order_date_str#11, Some(dd/MM/yyyy), Some(Etc/UTC), true), to_date(order_date_str#11, Some(yyyy/MM/dd), Some(Etc/UTC), true), to_date(order_date_str#11, Some(dd-MM-yyyy), Some(Etc/UTC), true)) AS order_date_parsed#16]
                  +- Project [order_id#6, city#7, product#8, amount#3, order_date#4, status#13, order_date_str#11, amount_int#15]
                     +- Project [or